# 06b · Prompt Engineering: What Attention Math Does and Doesn't Explain

**Time:** about 45 minutes · **Prerequisites:** `06_Attention_Mechanism`

## Learning objectives
1. Derive **softmax dilution**: why one relevant token gets less attention as
   context grows, and by exactly how much.
2. See why similarity between query and key drives attention, which is
   the reason relevant retrieved context (RAG) helps.
3. Show that softmax itself has **no position bias**, so effects like
   "lost in the middle" must come from what a model *learned*.
4. **Measure** a position effect on a real LLM with a needle-in-a-haystack test.

Parts 1–3 run offline in seconds. Part 4 uses
`mlx-community/Qwen2.5-3B-Instruct-4bit` (the notebook 08 model, about 1.7 GB
on first download) and is switched off by default.

In [ ]:
import numpy as np
import mlx.core as mx
import matplotlib.pyplot as plt

from mlx_nlp_utils import print_device_info, sanity_check, scaled_dot_product_attention

print_device_info()

## Part 1: Softmax dilution

Suppose one key in the context is relevant: its score beats every other
key's by a margin $m$. With $N$ keys in total, the attention on the relevant
key is

$$ w = \frac{e^{m}}{e^{m} + (N-1)} $$

For fixed $m$, $w \to 0$ as $N$ grows: every irrelevant token takes a share
of a budget that must sum to 1.

🤔 **Predict:** with a margin of 4 (a strong match), what fraction of attention
does the relevant key get among 1,000 tokens?

In [ ]:
lengths = np.array([10, 30, 100, 300, 1000, 3000, 10000])
plt.figure(figsize=(8, 4))
for m in [2, 4, 8]:
    plt.plot(lengths, np.exp(m) / (np.exp(m) + lengths - 1), 'o-', label=f'margin m = {m}')
plt.xscale('log'); plt.xlabel('context length N'); plt.ylabel('weight on the relevant key')
plt.title('Softmax dilution'); plt.legend(); plt.show()

# The same effect with real vectors. Key 0 equals the query, so its scaled
# score is about sqrt(d) = 8; random keys score about N(0, 1). Since the average
# of e^s for s ~ N(0, 1) is e^0.5, the effective margin is about 8 - 0.5 = 7.5.
mx.random.seed(0)
d = 64
for n in [10, 100, 1000, 10000]:
    q = mx.random.normal((1, 1, d))
    q = q / mx.linalg.norm(q) * np.sqrt(d)                                   # |q|^2 = d exactly
    k = mx.concatenate([q, mx.random.normal((1, n - 1, d))], axis=1)
    _, w = scaled_dot_product_attention(q, k, mx.zeros((1, n, d)))
    predicted = np.exp(7.5) / (np.exp(7.5) + n - 1)
    print(f"N={n:>6}: measured weight on the matching key {w[0, 0, 0].item():.3f}, formula {predicted:.3f}")

At a margin of 4, a relevant token gets about 5% of the attention among
1,000 tokens, and even the strong match above (margin ≈ 7.5) falls below
20% at 10,000 tokens. Real models cope through very large margins, many heads, and
many layers, but the arithmetic explains a practical rule: **padding a
prompt with irrelevant text is not free.**

## Part 2: Similarity drives attention

Attention scores are dot products, so keys that point the same way as the
query get more weight. Below, one key is the query plus a little noise, one
is the query plus more noise, and three are unrelated.

In [ ]:
mx.random.seed(42)
d = 8
query = mx.random.normal((1, 1, d))
keys = mx.concatenate([
    query + 0.1 * mx.random.normal((1, 1, d)),     # very similar
    mx.random.normal((1, 3, d)),                   # unrelated
    query + 0.5 * mx.random.normal((1, 1, d)),     # somewhat similar
], axis=1)
_, w = scaled_dot_product_attention(query, keys, mx.random.normal((1, 5, d)))

names = ["very similar", "unrelated 1", "unrelated 2", "unrelated 3", "somewhat similar"]
plt.figure(figsize=(9, 3.5))
bars = plt.bar(names, np.array(w[0, 0]), color=['green', 'gray', 'gray', 'gray', 'lightgreen'])
for bar, value in zip(bars, np.array(w[0, 0])):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 0.01, f'{value:.2f}', ha='center')
plt.ylabel('attention weight'); plt.title('Similar keys receive more attention'); plt.show()

This is the intuition behind RAG (notebook 09): put text that is *relevant
to the question* into the context, and the model has something to attend to.
One caution: in a real model the similarity is measured *after* the learned
projections $W_Q, W_K$ (notebook 06, Part 3), so "relevant" means relevant as
the model learned it, not necessarily as word overlap suggests.

## Part 3: Does softmax prefer the start or end of a prompt?

A common claim is that "attention favors the beginning and end of a prompt."
Let's test whether that comes from the attention **math**. Average the
attention a query gives to each position over many random trials:

🤔 **Predict:** will the average weight curve be U-shaped?

In [ ]:
mx.random.seed(0)
trials, n, d = 2000, 20, 16
q = mx.random.normal((trials, 1, d))
k = mx.random.normal((trials, n, d))
_, w = scaled_dot_product_attention(q, k, mx.zeros((trials, n, d)))
mean_w = np.array(w[:, 0, :].mean(axis=0))

plt.figure(figsize=(9, 3.5))
plt.bar(range(n), mean_w, color='steelblue')
plt.axhline(1 / n, color='black', linestyle='--', label='uniform 1/N')
plt.xlabel('position in context'); plt.ylabel('mean attention weight'); plt.legend()
plt.title('Average attention by position, 2,000 random trials'); plt.show()
print(f"Spread across positions: min {mean_w.min():.4f}, max {mean_w.max():.4f} (uniform {1 / n:.4f})")
sanity_check(mean_w.max() - mean_w.min() < 0.01, "softmax attention alone has no position bias")

Flat. The formula treats every position the same; **nothing in softmax
attention prefers the edges of a prompt.** (A single random trial can look
peaked somewhere, which is why one plot of one trial proves nothing.)

When real models do show position effects, as Liu et al. (2023) reported
in "Lost in the Middle", the cause is what the model **learned**: positional
encodings, training data where key information tends to appear at the start
or end, and attention "sinks" on the first tokens. That makes it an
empirical question about a specific model. Part 4 measures it.

## Part 4: Measure a position effect on a real model

**Needle in a haystack:** hide one fact (the "needle") at different depths
inside a long block of unrelated text, ask for it, and record accuracy by
depth. The filler is the knowledge base from notebook 09, repeated and
shuffled.

Set `RUN_REAL_MODEL = True` to run it. The first run downloads about 1.7 GB;
the default experiment takes a few minutes. Make the task harder with
`haystack_copies` (longer context) and `n_distractors` (similar-looking but
wrong needles). **Prepare for a null result:** in our runs on this model the
needle was found every time, as the interpretation below explains.

In [ ]:
RUN_REAL_MODEL = False

if RUN_REAL_MODEL:
    import random
    from mlx_lm import generate, load
    from mlx_lm.sample_utils import make_sampler
    from mlx_nlp_utils import load_rag_knowledge_base

    llm, tokenizer = load("mlx-community/Qwen2.5-3B-Instruct-4bit")
    greedy = make_sampler(temp=0.0)
    filler_docs = [d for d in load_rag_knowledge_base() if not d.startswith("Synthetic")]
    depths = [0.0, 0.25, 0.5, 0.75, 1.0]
    trials_per_depth = 8
    haystack_copies = 4        # prompt length: about 400 tokens per copy. Try 16 (≈6K tokens) or 48 (≈19K; slow)
    n_distractors = 0          # wrong-but-similar needles, e.g. "The access code for the garage is 4821."
    rng = random.Random(0)
    rooms = ["storage room", "server room", "front gate", "archive", "loading dock", "roof", "garage", "mail room"]

    def needle_trial(depth):
        code = str(rng.randint(1000, 9999))
        docs = filler_docs * haystack_copies
        rng.shuffle(docs)
        for room in rng.sample(rooms, n_distractors):
            docs.insert(rng.randrange(len(docs)), f"The access code for the {room} is {rng.randint(1000, 9999)}.")
        docs.insert(round(depth * len(docs)), f"The access code for the lab is {code}.")
        messages = [{"role": "user", "content": "\n".join(docs) +
                     "\n\nWhat is the access code for the lab? Reply with the number only."}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        answer = generate(llm, tokenizer, prompt=prompt, max_tokens=8, sampler=greedy)
        return code in answer, len(tokenizer.encode(prompt))

    accuracy = {}
    for depth in depths:
        outcomes = [needle_trial(depth) for _ in range(trials_per_depth)]
        accuracy[depth] = np.mean([ok for ok, _ in outcomes])
        print(f"depth {depth:.2f}: {accuracy[depth]:.0%} correct (prompt ≈ {outcomes[0][1]} tokens)")

    plt.figure(figsize=(7, 3.5))
    plt.plot(depths, [accuracy[d] for d in depths], 'o-')
    plt.ylim(-0.05, 1.05); plt.xlabel('needle depth (0 = start, 1 = end)'); plt.ylabel('accuracy')
    plt.title(f'Needle in a haystack, {trials_per_depth} trials per depth'); plt.show()
else:
    print("Skipped. Set RUN_REAL_MODEL = True to download the model and run the experiment.")

**What we measured** (Qwen2.5-3B-Instruct, greedy decoding):

| Haystack | Needle | Result |
|---|---|---|
| ≈1.6K tokens | lab code only | 100% at every depth (8 trials per depth) |
| ≈6K tokens | lab code only | 100% at depths 0, 0.5, 1 (4 trials each) |
| ≈19K tokens | lab code only | 100% at depths 0, 0.5, 1 (4 trials each; about 2.5 min per depth) |
| ≈6K tokens | lab code + 6 similar distractor codes | 100% except one miss in 6 at depth 0.5 (5 depths, 6 trials each) |

**This model shows no position effect on this task.** That does not
contradict "Lost in the Middle": that work used different models and
multi-document question answering, and our needle is **conspicuous**, a
sentence that looks nothing like the surrounding documents, which makes it
easy to find from anywhere. Even with similar-looking distractors, the single
miss (1 of 30) is well within what chance and a small sample explain.

Three honest takeaways:

1. A position effect is a property of a **specific model on a specific
   task**, not a law. Part 3 showed it cannot come from softmax itself.
2. With 4–8 trials per cell, one trial is 12–25 points; do not read a single
   miss as a trend.
3. To find a model's limits, push harder: more distractors, needles that
   resemble the filler, several needles, a smaller model (try
   `mlx-community/Qwen2.5-0.5B-Instruct-4bit`), or much longer contexts.
   Costs grow quickly with length, which is itself a lesson about long-context inference.

## Cheat sheet: which prompting advice rests on what

| Advice | Basis | How sure? |
|---|---|---|
| Remove irrelevant text | Softmax dilution (Part 1): every token takes a share of attention | Follows from the math |
| Retrieve relevant context (RAG) | Similar keys get more attention (Part 2) | Follows from the math, *in the model's learned space* |
| Put key instructions at the start or end | Learned position effects (Parts 3–4); not seen for this model on our needle task | Empirical: measure on your model |
| Ask for step-by-step reasoning | Causal models can only use tokens already generated, so writing intermediate steps gives later tokens more to attend to | Empirically helpful on many reasoning tasks |
| Use consistent terminology | Identical tokens have identical embeddings, so their keys match more easily | Plausible, model-dependent |

## 🧠 Exercises

**1. Dilution vs margin.** Using the formula in Part 1, what margin $m$ keeps
the relevant key above 50% weight with $N = 10{,}000$ tokens?

<details><summary>Answer</summary>

Solve $e^m = N - 1$: $m = \ln(9999) \approx 9.2$. Long-context models need very
sharp attention (large score margins) to single out one token, which is part
of why long-context training is hard.
</details>

**2. Instruction position.** Adapt Part 4: put the instruction "Reply in
French." at the start, middle, or end of a long prompt, and measure how often
the reply is in French. Is the effect the same as for retrieving a fact?

<details><summary>What to look for</summary>

Following an instruction and retrieving a fact can have different position
sensitivities. Instruction-tuned chat models are trained with instructions
in particular places (often the start of the user turn, or a system prompt),
so they may follow start-of-prompt instructions more reliably than buried ones.
</details>

**Next:** `07_Build_NanoGPT` builds the full Transformer around this attention mechanism.